# Import data and libraries

Welcome to the Aging clocks training notebook. Start from the necessary imports and read a short summary of aging clocks theoretical backgrounds below. 

Enjoy your clock building progress!

In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


First let's fetch a training dataset. For that we will use **nicely** working `huggingface-hub` lib. Requires approximately 4 minutes for downloading.

In [5]:
#Run this cell and magic happens
snapshot_download(
                repo_id='computage/computage_bench', 
                repo_type="dataset",
                local_dir='/tank/projects/computage/benchmarking_jan2025', #<---- choose your local directory for dataset
                allow_patterns=['computage_train_meta.tsv', 
                                'data/train/*.parquet'
                                ]
                )

Fetching 47 files:   0%|          | 0/47 [00:00<?, ?it/s]

computage_train_data_GSE235717.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

computage_train_data_GSE210245.parquet:   0%|          | 0.00/67.0M [00:00<?, ?B/s]

'/tank/projects/computage/benchmarking_jan2025'

We downloaded a lot of data, but for demonstrational purposes we will use one of them (but you can use all if you want!). Choose the largest one.

In [29]:
data_path = '/tank/projects/computage/benchmarking_jan2025/data/train/computage_train_data_GSE147740.parquet'
meta_path = '/tank/projects/computage/benchmarking_jan2025/computage_train_meta.tsv'
data = pd.read_parquet(data_path).T 
meta = pd.read_csv(meta_path, sep='\t', index_col=0).loc[data.index]

print('Data shape', data.shape)

Data shape (1024, 761653)


# First generation aging clocks training

Let's consider the typical procedure of aging clocks training on data of healthy individuals. Before starting the training, we **highly recommend** to read carefully the below assumptions and limitations of first-generation aging clocks (AC-I) to better understand the limits of their applicability.

### Definitions
Remember that biological age (BA, $B$) is typically defined as a generalized measure of human health compared to the average health of individuals at a given age within a population. Thus, if an individual has a biological age of 40 at the chronological age (CA, $C$) of 30, it is assumed that their overall health corresponds to that of an average 40-year-old in the population. This relationship can be concisely expressed as:
$$ B = C + \Delta ,$$ 
where $\Delta$ symbolizes BA acceleration (or deceleration if negative). In general, BA can be estimated from a set of biomarkers $X$ with a model (algorithm) $f : X \rightarrow B$ also called **aging clock**. However, BA is *latent*: it has no ground truth value that can be measured directly and then used to train an aging clock model $f$ in a classical supervised fashion, making clock validation a nontrivial task.

First generation clocks stems from the idea that chronological age can be a "good approximation" (or proxy) for the biological age. The concept of biological age is substituted with the concept of chronological age as follows:

1) Model $f$ is trained to predict chronological age with respect to the classic regression task: $C = \hat{C} + \varepsilon = f(X) + \varepsilon$. 
2) Model predictions $\hat{C}$ are denoted as Biological Age, i.e.: $B = \hat{C}$. (<- that is where substitution of concepts occurs).

Importantly, first generation clocks can be derived from (trained on) any type of biomarkers $X$: blood parameters, physiological examinations, psychological assays, DNA methylation profiles, photographs, etc. In this notebook we consider only the case of training clocks on DNA methylation data. 

### Paradox of biomarkers

This definition has an important corollary, namely $\varepsilon = -\Delta$. This means that  *zero model error $\varepsilon \rightarrow 0$ implies zero age acceleration $\Delta \rightarrow -0$*. This situation is called "The paradox of biomarkers" and corresponds to the situation of training a *perfect* regression model with naught error. However, despite this limitation, sometimes AC-I clocks returns reasonable results and because the actual prediction error $\varepsilon$ can be *correctly* associated with unobservable age acceleration $\Delta$. 

### Identical association assumption

Why first generation clocks works? The formidable theoretical answer to this questions is presented in this [article](https://bmcmedresmethodol.biomedcentral.com/articles/10.1186/s12874-024-02181-x). Here we briefly repeat the main idea of the article. Clocks correctly predict acceleration (or deceleration) of biological age if underlying biomarkers ($X$), that were used for training, are associated with Aging-Acceleration (Deceleration) in the same direction. For example, suppose you decided to use systolic blood pressure level for training clock. We know this level increases gradually with age in average in population. On the other hand, we **assume** that blood pressure is even higher in more biologically aged persons (e.g. persons suffering from age-related diseases). This positive association between blood pressure levels and chronological age along with the positive association of blood pressure and some age related disease ensures that clocks trained on blood pressure will predict positive age acceleration for unhealthy patients. This last statement is called *Identical association assumption*.

# Too much theory, let's start training

Ok, let's do it! Let's train a simple aging clocks model using *all* data we have.

In [30]:
from sklearn.linear_model import LinearRegression, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

y = meta['Age'].copy()
X = data.copy()

#let's split the data to train and validation set
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=10)

Methylation data are typically contain a huge number of features (CpG sites). So, it is reasonable to conduct a feature selection before training. Let's sort features by their correlation with target variable - chronological age. Note, the below cell can take a couple of minutes to compute.

In [31]:
pcorr = X_train.corrwith(y_train, method='pearson')

/home/dkriukov/.conda/envs/computage/lib/python3.10/site-packages/numpy/lib/function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/home/dkriukov/.conda/envs/computage/lib/python3.10/site-packages/numpy/lib/function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/dkriukov/.conda/envs/computage/lib/python3.10/site-packages/numpy/lib/function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/home/dkriukov/.conda/envs/computage/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/dkriukov/.conda/envs/computage/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


There are a lot of feature well correlated with chronological age, so we can afford ourselves to choose quite strict filter for them.

In [35]:
#choose features with absolute correlation with age higher than some threshold
selected_features = pcorr[np.abs(pcorr) > 0.3].index 
X_train_ = X_train[selected_features]
X_val_ = X_val[selected_features]

print('Shape after filtering', X_train_.shape)

Shape after filtering (870, 3073)


Now let's drop features with NaNs in train and simply fill them with 0.5 in validation set. Note that, it is not common strategy. Filling NaNs with averages or external methylation beta values can be more beneficial. 

In [51]:
X_train_ = X_train_.dropna(axis=1)
X_val_ = X_val_[X_train_.columns].fillna(0.5)

We want something simple yet robust. It is good practice to compress methylation data before training, because it saves compute time and gives good results. We will use `Pipeline` class for that to gather preprocessing step and model in one object.

In [52]:
model = Pipeline([
                ('pca', PCA(150)),
                ('lr', LinearRegression()),
                ])

model.fit(X_train_, y_train)

Pipeline(steps=[('pca', PCA(n_components=150)), ('lr', LinearRegression())])

It is common to measure clock performance in terms of coefficient of determination ($R^2$ score) or mean absolute error (MAE).

In [53]:
y_pred_train = model.predict(X_train_)
y_pred_val = model.predict(X_val_)

mae_train = mean_absolute_error(y_train, y_pred_train)
mae_val = mean_absolute_error(y_val, y_pred_val)
r2_train = r2_score(y_train, y_pred_train)
r2_val = r2_score(y_val, y_pred_val)

print(f'Train set performance metrics: MAE={round(mae_train, 3)}, R2={round(r2_train, 3)}')
print(f'Validation set performance metrics: MAE={round(mae_val, 3)}, R2={round(r2_val, 3)}')

Train set performance metrics: MAE=1.402, R2=0.947
Validation set performance metrics: MAE=1.933, R2=0.872


We slightly overfitted but received quite good results. I hope this notebook will be a good starting point for you to explore *the realm of aging clocks*.

# Credentials

The notebook was prepared by [Dmitrii Kriukov](https://scholar.google.com/citations?user=Wo9H1f4AAAAJ&hl=ru)